# Figure 2 — pages
Assembles the panels drawn by `tgfb_figures.ipynb` into the pages (laid out on A4, then cropped to the content):
- **Figure 2** — a schematic (vector, `data/fig2a_schematic.svg`); b ECM pathway contact graphs (young-only | other |
  old-only | other); c TGF-β response of genes forming age-specific contacts vs other genes; d promoter accessibility.
- **Supplementary Figure 2Sa** — a expression histograms; b contact graphs for TGF-β signaling, senescence inhibitors and
  MYC targets; c promoter-peak change under TGF-β.
- **Supplementary Figure 2Sb** — a locus gene density by contact group; b TGF-β response by contact group, young vs old.

Panels are nested as vector SVG (text stays editable). Page SVGs go to `../../illustrator-figures/`, PDF/PNG to
`figures/`. Needs `cairosvg`.

In [1]:
import copy
import io
import xml.etree.ElementTree as ET

import cairosvg
import numpy as np
from matplotlib.font_manager import FontProperties
from matplotlib.textpath import TextPath
from PIL import Image

FIG = "figures"
PAGES = "../../illustrator-figures"                    # page SVGs live with the Illustrator files
SVG_NS = "http://www.w3.org/2000/svg"
ARIAL = FontProperties(family="Arial")
for prefix, uri in [("", SVG_NS), ("xlink", "http://www.w3.org/1999/xlink"),
                    ("rdf", "http://www.w3.org/1999/02/22-rdf-syntax-ns#"), ("cc", "http://creativecommons.org/ns#"),
                    ("dc", "http://purl.org/dc/elements/1.1/")]:
    ET.register_namespace(prefix, uri)
PAGE_W, PAGE_H = 595.276, 841.89                       # A4 (pt)
CONTENT_X, CONTENT_W = 28, 543                         # panel area left edge / width
GAP_Y = 20                                             # vertical gap between panels (pt)
PAD = 4                                                # margin kept around the content when cropping (pt)
# Figure 2 palette for page text and legend swatches (graph edges / KDE hues; the KDE "other" curves are #9c9c9c)
YOUNG, OLD, GREY = "#1f77b4", "#d62728", "#9e9e9e"


def load(path):
    """Root element of an SVG file."""
    return ET.parse(path).getroot()


def size(root):
    """(width, height) from the viewBox."""
    return tuple(float(v) for v in root.get("viewBox").split()[2:])


def new_page():
    """Blank A4 page with a white background."""
    page = ET.Element(f"{{{SVG_NS}}}svg", width=f"{PAGE_W}pt", height=f"{PAGE_H}pt", viewBox=f"0 0 {PAGE_W} {PAGE_H}")
    ET.SubElement(page, f"{{{SVG_NS}}}rect", id="background", x="0", y="0", width=f"{PAGE_W}", height=f"{PAGE_H}",
                  fill="#ffffff")
    return page


_nested = [0]                                          # panels nested so far (unique id prefixes)


def prefix_ids(root, prefix):
    """Make every id in a nested SVG unique (and update its references), so panels converted by the same tool
    (e.g. pdftocairo's glyph0-0 ...) don't collide on one page."""
    ids = {e.get("id") for e in root.iter() if e.get("id")}
    for e in root.iter():
        if e.get("id"):
            e.set("id", prefix + e.get("id"))
        for attr, val in list(e.attrib.items()):
            if "#" not in val:
                continue
            if val.startswith("#") and val[1:] in ids:              # href="#id"
                e.set(attr, f"#{prefix}{val[1:]}")
            elif "url(#" in val:                                      # fill / clip-path / mask="url(#id)"
                for i in ids:
                    val = val.replace(f"url(#{i})", f"url(#{prefix}{i})")
                e.set(attr, val)


def nest(page, root, x, y, scale, viewbox=None):
    """Place an SVG on the page at (x, y), scaled; optional viewBox crop (x, y, w, h). Returns its bottom."""
    vb = viewbox or tuple(float(v) for v in root.get("viewBox").split())
    child = ET.SubElement(page, f"{{{SVG_NS}}}svg", x=f"{x}", y=f"{y}", width=f"{vb[2] * scale}",
                          height=f"{vb[3] * scale}", viewBox=" ".join(f"{v}" for v in vb),
                          preserveAspectRatio="xMinYMin meet", overflow="hidden")
    _nested[0] += 1
    root = copy.deepcopy(root)
    prefix_ids(root, f"p{_nested[0]}_")
    for el in root:
        child.append(el)
    return y + vb[3] * scale


def text(page, s, x, y, size=12, color="#000000", anchor="start", weight="normal"):
    """Text in Arial at (x, y)."""
    t = ET.SubElement(page, f"{{{SVG_NS}}}text", x=f"{x}", y=f"{y}", style=(
        f"font-family: Arial, ArialMT, sans-serif; font-size: {size}px; fill: {color}; text-anchor: {anchor}; "
        f"font-weight: {weight}"))
    t.text = s


def letter(page, s, x, y):
    """Panel letter (Arial 12, lowercase + period)."""
    text(page, f"{s}.", x, y)


def full_width(page, name, root, y, max_scale=None, x=CONTENT_X, width=CONTENT_W):
    """Panel spanning the given width (optionally capped in scale), with its letter. Returns its bottom."""
    w, _ = size(root)
    scale = width / w if max_scale is None else min(width / w, max_scale)
    letter(page, name, CONTENT_X - 11, y + 8)
    return nest(page, root, x, y, scale)


def crop_to_content(page):
    """Shrink the page viewBox (and background) to the drawn content + PAD, so the page has no white margins."""
    png = cairosvg.svg2png(bytestring=ET.tostring(page), scale=4)
    ink = np.asarray(Image.open(io.BytesIO(png)).convert("L")) < 250
    px = ink.shape[1] / PAGE_W                                   # pixels per pt
    rows = np.where(ink.any(axis=1))[0]
    cols = np.where(ink.any(axis=0))[0]
    x0, x1 = max(cols[0] / px - PAD, 0), min((cols[-1] + 1) / px + PAD, PAGE_W)
    y0, y1 = max(rows[0] / px - PAD, 0), min((rows[-1] + 1) / px + PAD, PAGE_H)
    page.set("viewBox", f"{x0:.2f} {y0:.2f} {x1 - x0:.2f} {y1 - y0:.2f}")
    page.set("width", f"{x1 - x0:.2f}pt")
    page.set("height", f"{y1 - y0:.2f}pt")
    page.find(f"{{{SVG_NS}}}rect[@id='background']").attrib.update(
        x=f"{x0:.2f}", y=f"{y0:.2f}", width=f"{x1 - x0:.2f}", height=f"{y1 - y0:.2f}")
    return x1 - x0, y1 - y0


def write(page, name, bottom):
    """Crop the page to its content; save the SVG next to the Illustrator files and PDF + PNG previews in figures/."""
    w, h = crop_to_content(page)
    print(f"{name}: cropped to {w:.0f} x {h:.0f} pt")
    ET.ElementTree(page).write(f"{PAGES}/{name}.svg", xml_declaration=True, encoding="utf-8")
    cairosvg.svg2pdf(url=f"{PAGES}/{name}.svg", write_to=f"{FIG}/{name}.pdf")
    cairosvg.svg2png(url=f"{PAGES}/{name}.svg", write_to=f"{FIG}/{name}.png", scale=2)
    print(f"{name}: layout bottom (before cropping) at {bottom:.0f} of {PAGE_H:.0f} pt")


def line_legend(page, items, x, y, size=7, dy=10):
    """Legend of coloured line swatches with labels; items = [(label, colour)]."""
    for i, (label, color) in enumerate(items):
        yy = y + i * dy
        ET.SubElement(page, f"{{{SVG_NS}}}line", x1=f"{x}", y1=f"{yy - 2.5}", x2=f"{x + 12}", y2=f"{yy - 2.5}",
                      stroke=color, **{"stroke-width": "1.6"})
        text(page, label, x + 16, yy, size=size)


def text_width(s, size):
    """Rendered width (pt) of a string in Arial at the given size."""
    return TextPath((0, 0), s, size=size, prop=ARIAL).get_extents().width


def rich_text(page, parts, x, y, size=9):
    """One centred line built from (string, colour) parts, e.g. a coloured word followed by black text."""
    widths = [text_width(s.replace(" ", "_"), size) for s, _ in parts]   # "_" ~ a space's advance, keeps spacing
    left = x - sum(widths) / 2
    for (s, color), w in zip(parts, widths):
        text(page, s.strip(), left + (text_width("_", size) if s.startswith(" ") else 0), y, size=size, color=color)
        left += w


def bracket(page, x0, x1, y, h=6):
    """Open bracket under a pair of panels: a horizontal line with short upward ticks at both ends."""
    ET.SubElement(page, f"{{{SVG_NS}}}path", d=f"M {x0} {y - h} L {x0} {y} L {x1} {y} L {x1} {y - h}",
                  fill="none", stroke="#000000", **{"stroke-width": "1.2"})


def subset_titles(page, x, width, y, fracs, size=9):
    """Titles over the four contact-subset graphs: young-only | other | old-only | other."""
    for frac, parts in zip(fracs, [[("Young-only", YOUNG), (" subset", "#000000")], [("Other subset", "#000000")],
                                   [("Old-only", OLD), (" subset", "#000000")], [("Other subset", "#000000")]]):
        rich_text(page, parts, x + width * frac, y, size=size)


EDGE_LEGEND = load(f"{FIG}/fig2_legend_edge_types.svg")
GRAPH_FRACS = (0.115, 0.355, 0.59, 0.82)               # centres of the four graphs within a subset-graph SVG

## Figure 2

In [2]:
page = new_page()
# a: experiment schematic (hand-drawn vector, data/fig2a_schematic.svg)
bottom = nest(page, load("data/fig2a_schematic.svg"), 30, 8, 1.0)
letter(page, "a", CONTENT_X - 11, 16)

# b: ECM pathway contact graphs: pathway name + edge legend on the left; young-only | other | old-only | other subsets
# with titles above and a bracket under each pair
y = bottom + GAP_Y
letter(page, "b", CONTENT_X - 11, y + 8)
B_LEFT = 96
b_root = load(f"{FIG}/fig2b_ecm_contact_graphs.svg")
b_x = CONTENT_X + B_LEFT
b_w = CONTENT_W - B_LEFT
b_y = y + 16
subset_titles(page, b_x, b_w, b_y - 4, GRAPH_FRACS)
b_bottom = nest(page, b_root, b_x, b_y, b_w / size(b_root)[0])
for left, right in ((0, 1), (2, 3)):                   # bracket centred under each pair of graphs
    mid = (GRAPH_FRACS[left] + GRAPH_FRACS[right]) / 2
    half = 0.17
    bracket(page, b_x + b_w * (mid - half), b_x + b_w * (mid + half), b_bottom + 6)
text(page, "Extracellular matrix", CONTENT_X + B_LEFT / 2 - 6, b_y + 10, size=9, anchor="middle")
text(page, "pathway", CONTENT_X + B_LEFT / 2 - 6, b_y + 21, size=9, anchor="middle")
nest(page, EDGE_LEGEND, CONTENT_X + 6, b_y + 34, 0.62)
bottom = b_bottom + 8

# c: TGF-β response of genes forming age-specific contacts vs other genes (rows: young / old fibroblasts)
letter(page, "c", CONTENT_X - 11, bottom + GAP_Y + 8)
y = bottom + GAP_Y + 14                                # panel and its legends sit below the letter
LEFT = 78                                              # left column for row labels and legends
c_root = load(f"{FIG}/fig2c_tgfb_response.svg")
c_bottom = nest(page, c_root, CONTENT_X + LEFT, y, (CONTENT_W - LEFT) / size(c_root)[0])
row_h = (c_bottom - y - 16) / 3                        # the shared x-axis label sits under the last row
for i, (label, color) in enumerate([("Young fibroblasts", YOUNG), ("Old fibroblasts", OLD), ("Old fibroblasts", OLD)]):
    cy = y + 10 + row_h * (i + 0.5)
    for j, word in enumerate(label.split()):
        text(page, word, CONTENT_X + 32, cy + 10 * j, size=9, color=color, anchor="middle")
line_legend(page, [("Young-specific contacts", YOUNG), ("No young-specific", GREY)], CONTENT_X - 4, y + 14, size=6.5, dy=9)
line_legend(page, [("Old-specific contacts", OLD), ("No old-specific", GREY)], CONTENT_X - 4, y + 10 + 2 * row_h + 4,
            size=6.5, dy=9)

# d: promoter accessibility: % of young-specific vs other pathway genes with a promoter peak, control vs TGF-β
y = c_bottom + GAP_Y
d_root = load(f"{FIG}/fig2d_promoter_accessibility.svg")
bottom = nest(page, d_root, CONTENT_X + LEFT, y, (CONTENT_W - LEFT) / size(d_root)[0])
letter(page, "d", CONTENT_X - 11, y + 8)
text(page, "Young", CONTENT_X + 32, (y + bottom) / 2 - 6, size=9, color=YOUNG, anchor="middle")
text(page, "fibroblasts", CONTENT_X + 32, (y + bottom) / 2 + 4, size=9, color=YOUNG, anchor="middle")
write(page, "Figure2", bottom)

Figure2: cropped to 561 x 748 pt


Figure2: layout bottom (before cropping) at 755 of 842 pt


## Supplementary Figure 2Sa

In [3]:
page = new_page()
bottom = full_width(page, "a", load(f"{FIG}/fig2Sa_a_expression_histograms.svg"), 17, max_scale=0.5)

# b: contact subsets of the other three young-colocalized pathways: young-only | other | old-only | other
y = bottom + GAP_Y
LABEL_W = 60
nest(page, EDGE_LEGEND, CONTENT_X - 4, y + 4, 0.55)
letter(page, "b", CONTENT_X - 11, y + 8)               # after the legend, so its frame does not cover the period
g_y = y + 40
g_root = load(f"{FIG}/fig2Sa_b_contact_graphs.svg")
g_x = CONTENT_X + LABEL_W
g_w = CONTENT_W - LABEL_W
subset_titles(page, g_x, g_w, g_y - 4, GRAPH_FRACS, size=8)
g_bottom = nest(page, g_root, g_x, g_y, g_w / size(g_root)[0])
for i, name in enumerate(["TGF-β signaling", "Senescence inhibitors", "MYC targets"]):
    cy = g_y + (g_bottom - g_y) * (i + 0.5) / 3
    for j, word in enumerate(name.split(" ", 1)):
        text(page, word, CONTENT_X + LABEL_W / 2 - 6, cy + 10 * j, size=8, color=YOUNG, anchor="middle")

# c: promoter accessibility change under TGF-β (DESeq2 log2FC of promoter peaks), young-specific vs other genes
y = g_bottom + GAP_Y
c_root = load(f"{FIG}/fig2Sa_c_promoter_peak_de.svg")
LEFT = 60
bottom = nest(page, c_root, CONTENT_X + LEFT, y, (CONTENT_W - LEFT) / size(c_root)[0])
letter(page, "c", CONTENT_X - 11, y + 8)
text(page, "Young", CONTENT_X + LEFT / 2 - 6, (y + bottom) / 2 - 6, size=8, color=YOUNG, anchor="middle")
text(page, "fibroblasts", CONTENT_X + LEFT / 2 - 6, (y + bottom) / 2 + 4, size=8, color=YOUNG, anchor="middle")
write(page, "Figure2Sa", bottom)

Figure2Sa: cropped to 561 x 732 pt


Figure2Sa: layout bottom (before cropping) at 745 of 842 pt


## Supplementary Figure 2Sb
a and b stacked at full width, each at its drawn size (never enlarged).

In [4]:
page = new_page()
y = 17
for tag, name in (("a", "fig2Sb_a_gene_density"), ("b", "fig2Sb_b_response_by_contact_group")):
    root = load(f"{FIG}/{name}.svg")
    letter(page, tag, CONTENT_X - 11, y + 8)
    y = nest(page, root, CONTENT_X, y, min(1.0, CONTENT_W / size(root)[0])) + 18
write(page, "Figure2Sb", y)

Figure2Sb: cropped to 520 x 662 pt


Figure2Sb: layout bottom (before cropping) at 698 of 842 pt
